In [1]:
# Importar dados
# ragas — avalia a qualidade do pipeline RAG
# datasets — formato que o ragas espera receber

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision
)
from datasets import Dataset
import pandas as pd

print("✅ Imports prontos")

✅ Imports prontos


C:\Users\renat\AppData\Local\Temp\ipykernel_23404\1228147477.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\renat\AppData\Local\Temp\ipykernel_23404\1228147477.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\renat\AppData\Local\Temp\ipykernel_23404\1228147477.py:6: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
C:\Users\renat\AppData\Local\Temp\ipykernel_23404\12

In [2]:
# Carregar dados:
# O amnesty_qa já tem tudo que o RAGAS precisa:
# pergunta + resposta + contextos + resposta correta (ground truth)

from datasets import load_dataset

ds = load_dataset("explodinggradients/amnesty_qa", "english_v1", split="eval")

print(f"✅ {len(ds)} exemplos carregados")
print(f"Colunas disponíveis: {ds.column_names}")

✅ 20 exemplos carregados
Colunas disponíveis: ['question', 'ground_truths', 'answer', 'contexts']


In [3]:
# Ver um exemplo primeiro: perceber o que tem antes de medir
# 
# O RAGAS precisa de 4 coisas:
# 1. question — a pergunta
# 2. answer — a resposta gerada
# 3. contexts — lista de contextos recuperados
# 4. ground_truths — a resposta correcta

exemplo = ds[0]

print("PERGUNTA:")
print(exemplo["question"])
print("\nRESPOSTA:")
print(exemplo["answer"][:200], "...")
print("\nCONTEXTOS (quantos):", len(exemplo["contexts"]))
print("\nGROUND TRUTH:")
print(exemplo["ground_truths"][:200])

PERGUNTA:
What are the global implications of the USA Supreme Court ruling on abortion?

RESPOSTA:
The global implications of the USA Supreme Court ruling on abortion can be significant, as it sets a precedent for other countries and influences the global discourse on reproductive rights. Here are  ...

CONTEXTOS (quantos): 3

GROUND TRUTH:
["The global implications of the USA Supreme Court ruling on abortion are significant. The ruling has led to limited or no access to abortion for one in three women and girls of reproductive age in states where abortion access is restricted. These states also have weaker maternal health support, higher maternal death rates, and higher child poverty rates. Additionally, the ruling has had an impact beyond national borders due to the USA's geopolitical and cultural influence globally. Organizations and activists worldwide are concerned that the ruling may inspire anti-abortion legislative and policy attacks in other countries. The ruling has also hind

In [5]:
# Preparar o dataset para o RAGAS - formato específico - converter


# Usar só os primeiros 10 exemplos para começar

# Fix: ground_truth precisa ser string
# Fix: truncar contextos para não exceder limite do Groq

n = 10

def extrair_ground_truth(valor):
    if isinstance(valor, list):
        return " ".join(valor)
    return valor

def truncar_contextos(contextos, max_chars=500):
    # Trunca cada contexto para não exceder o limite do modelo
    return [ctx[:max_chars] for ctx in contextos]

eval_dataset = Dataset.from_dict({
    "question":     [ds[i]["question"] for i in range(n)],
    "answer":       [ds[i]["answer"][:500] for i in range(n)],
    "contexts":     [truncar_contextos(ds[i]["contexts"]) for i in range(n)],
    "ground_truth": [extrair_ground_truth(ds[i]["ground_truths"]) for i in range(n)],
})

print(f"✅ Dataset preparado com {len(eval_dataset)} exemplos")
print(f"Preview contexto: {eval_dataset['contexts'][0][0][:100]}")

✅ Dataset preparado com 10 exemplos
Preview contexto: - In 2022, the USA Supreme Court handed down a decision ruling that overturned 50 years of jurisprud


In [7]:
# Configurar LLM para o RAGAS
# O RAGAS precisa de um LLM para calcular algumas métricas (optamos pelo Groq)


import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas import RunConfig
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

load_dotenv()

# LLM — mesmo modelo que está no outro projeto
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0,
    n=1
)

# Embeddings — mesmo modelo do pipeline RAG
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Wrappers para o RAGAS entender os modelos
ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

print("✅ LLM e embeddings prontos")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ LLM e embeddings prontos


C:\Users\renat\AppData\Local\Temp\ipykernel_23404\4106886537.py:27: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)
C:\Users\renat\AppData\Local\Temp\ipykernel_23404\4106886537.py:28: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)


In [8]:
# Configurar métricas: para cada métrica qual LLM e embeddings 


#faithfulness.llm = ragas_llm
#answer_relevancy.llm = ragas_llm
#answer_relevancy.embeddings = ragas_embeddings
#context_recall.llm = ragas_llm
#context_precision.llm = ragas_llm

from ragas.metrics import answer_relevancy

metricas = [
    answer_relevancy]

print("✅ Métricas configuradas")

✅ Métricas configuradas


C:\Users\renat\AppData\Local\Temp\ipykernel_23404\561683142.py:10: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import answer_relevancy


In [10]:
# Rodar avaliação

print("A avaliar... pode demorar 3-5 minutos ☕")

results = evaluate(
    dataset=eval_dataset,
    metrics=metricas,
    llm=ragas_llm,  
    run_config=RunConfig(max_workers=1)
)

print("\n✅ Avaliação completa!")
print(results)

A avaliar... pode demorar 3-5 minutos ☕


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable